In [3]:
import pandas as pd 
from lets_plot import *
import sqlite3
import json
LetsPlot.setup_html()
import numpy as np 
import inspect
import plotly.express as px

In [4]:
conn = sqlite3.connect('../../data/main.db') 

fire_pokemon_sql_query = '''
SELECT *
  FROM fire_pokemon;
'''

ice_pokemon_sql_query = '''
SELECT * 
  FROM ice_pokemon;
'''

water_pokemon_sql_query = '''
SELECT * 
  FROM water_pokemon;
'''

fire_poke_df = pd.read_sql(fire_pokemon_sql_query, conn)
ice_pokemon_df = pd.read_sql(ice_pokemon_sql_query, conn)
water_poke_df = pd.read_sql(water_pokemon_sql_query, conn)

conn.close()


In [15]:
poke_df = pd.read_json('../../data/pokemon_data/main_pokemon_df.json')

In [6]:
custom_colors = {
    'grass': '#78C850',  # Green
    'fire': '#F08030',   # Red-Orange
    'water': '#6890F0',  # Blue
    'electric': '#F8D030',  # Yellow
    'ice': '#98D8D8',    # Light Blue
    'rock': '#B8A038',   # Brown
    'ground': '#E0C068', # Sandy Brown
    'steel': '#B8B8D0',  # Gray
    'fairy': '#EE99AC',  # Pink
    'ghost': '#705898',  # Purple
    'dragon': '#7038F8', # Deep Purple
    'dark': '#705848',   # Dark Gray
    'fighting': '#C03028',  # Red
    'psychic': '#F85888',  # Magenta
    'flying': '#A890F0',  # Light Blue
    'poison': '#A040A0',  # Purple
    'bug': '#A8B820',     # Olive Green
    'normal': '#A8A878',  # Neutral Beige
}

In [7]:
filtered_poke_df = poke_df[(poke_df['type_1'] == 'fire') | (poke_df['type_1'] == 'ice') | (poke_df['type_1'] == 'water')]
display(filtered_poke_df)

,name,url,generation,habitat_name,pokemon_description,type_1,type_2,pokemon_portrait,hp_stat,attack_stat,defense_stat,special_attack_stat,special_defense_stat,speed_stat,total_stat
4,charmander,https://pokeapi.co/api/v2/pokemon-species/4/,1,mountain,"Obviously prefers hot places. When it rains, s...",fire,None,https://raw.githubusercontent.com/PokeAPI/spri...,39,52,43,60,50,65,309
5,charmeleon,https://pokeapi.co/api/v2/pokemon-species/5/,1,mountain,"When it swings its burning tail, it elevates t...",fire,None,https://raw.githubusercontent.com/PokeAPI/spri...,58,64,58,80,65,80,405
6,charizard,https://pokeapi.co/api/v2/pokemon-species/6/,1,mountain,Spits fire that is hot enough to melt boulders...,fire,flying,https://raw.githubusercontent.com/PokeAPI/spri...,78,84,78,109,85,100,534
7,squirtle,https://pokeapi.co/api/v2/pokemon-species/7/,1,waters-edge,"After birth, its back swells and hardens into ...",water,None,https://raw.githubusercontent.com/PokeAPI/spri...,44,48,65,50,64,43,314
8,wartortle,https://pokeapi.co/api/v2/pokemon-species/8/,1,waters-edge,Often hides in water to stalk unwary prey. For...,water,None,https://raw.githubusercontent.com/PokeAPI/spri...,59,63,80,65,80,58,405
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
977,dondozo,https://pokeapi.co/api/v2/pokemon-species/977/,9,None,"This Pokémon is a glutton, but it’s bad at get...",water,None,https://raw.githubusercontent.com/PokeAPI/spri...,150,100,115,65,65,35,530
991,iron-bundle,https://pokeapi.co/api/v2/pokemon-species/991/,9,None,Its shape is similar to a robot featured in a ...,ice,water,https://raw.githubusercontent.com/PokeAPI/spri...,56,80,114,124,60,136,570
994,iron-moth,https://pokeapi.co/api/v2/pokemon-species/994/,9,None,This Pokémon resembles an unknown object descr...,fire,poison,https://raw.githubusercontent.com/PokeAPI/spri...,80,70,60,140,110,110,570
1009,walking-wake,https://pokeapi.co/api/v2/pokemon-species/1009/,9,None,This ferocious creature is shrouded in mystery...,water,dragon,https://raw.githubusercontent.com/PokeAPI/spri...,99,83,91,125,83,109,590


In [8]:
display(poke_df.iloc[0,7])

'https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/1.png'

In [9]:
import requests
import numpy as np
from sklearn.cluster import KMeans
from PIL import Image
from io import BytesIO

def get_main_color_from_url(image_url):
    # Fetch the image from the URL
    response = requests.get(image_url)
    img = Image.open(BytesIO(response.content))
    
    # Convert image to RGB (if not already in RGB mode)
    img_rgb = img.convert('RGB')
    
    # Convert image to numpy array
    img_array = np.array(img_rgb)
    
    # Threshold to detect white background (pure white pixels)
    lower_white = np.array([200, 200, 200])  # Lower threshold for white
    upper_white = np.array([255, 255, 255])  # Upper threshold for white
    
    # Create a mask to find white pixels (background)
    mask = np.all(np.logical_and(img_array >= lower_white, img_array <= upper_white), axis=-1)
    
    # Invert mask to focus on the Pokémon (non-background pixels)
    mask_inv = ~mask
    
    # Mask the image to isolate the Pokémon (remove white background)
    img_no_bg = img_array[mask_inv]
    
    # Apply K-means clustering to find the dominant color from the remaining pixels
    kmeans = KMeans(n_clusters=1, random_state=0).fit(img_no_bg)
    
    # The center of the cluster is the main color
    main_color = kmeans.cluster_centers_[0]
    
    # Return the main color as an integer tuple
    return tuple(main_color.astype(int))

# Test the function with a URL
image_url = 'https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/1.png'
main_color = get_main_color_from_url(image_url)
print(f"Main color: {main_color}")



Main color: (7, 12, 10)


In [10]:
main_ridge_plot = ggplot(poke_df, aes(x='total_stat', y='type_1', fill='type_1')) + \
       geom_area_ridges(scale=4, alpha=0.7, tooltips=layer_tooltips().title("@type_1").line("Total Stat|@total_stat")) + \
       scale_fill_manual(values = custom_colors) + \
       ggtitle('Correlation between Primary Type and Stat Total',) + \
       xlab('Stat Total') + \
       ylab('Pokemon Type') + \
       theme_minimal() + \
       ggsize(1000, 600) + \
       theme(axis_text_x=element_text(angle=45, hjust=1),
             plot_title=element_text(size=16, face='bold', hjust = 0.5),
             axis_title=element_text(size=12),
             legend_title=element_text(size=12),
             legend_text=element_text(size=10))

main_ridge_plot.show()
ggsave(main_ridge_plot, "../../../visuals/html_files/main_ridge_plot.html")

'c:\\Users\\chang\\DS105A\\ds105a-2024-project-error_105\\visuals\\html_files\\main_ridge_plot.html'

In [11]:
filtered_ridge_plot = ggplot(filtered_poke_df, aes(x='total_stat', y='type_1', fill='type_1')) + \
       geom_area_ridges(scale=4, alpha=0.7, tooltips=layer_tooltips().title("@type_1").line("Total Stat|@total_stat")) + \
       scale_fill_manual(values = custom_colors) + \
       ggtitle('Correlation between Primary Type and Stat Total',) + \
       xlab('Stat Total') + \
       ylab('Pokemon Type') + \
       theme_minimal() + \
       ggsize(1000, 600) + \
       theme(axis_text_x=element_text(angle=45, hjust=1),
             plot_title=element_text(size=16, face='bold', hjust = 0.5),
             axis_title=element_text(size=12),
             legend_title=element_text(size=12),
             legend_text=element_text(size=10))

filtered_ridge_plot.show()
ggsave(filtered_ridge_plot, "../../../visuals/html_files/filtered_ridge_plot.html")

'c:\\Users\\chang\\DS105A\\ds105a-2024-project-error_105\\visuals\\html_files\\filtered_ridge_plot.html'

In [12]:
poke_df = pd.read_json('../../data/pokemon_data/main_pokemon_df.json')

poke_df['habitat_name'] = poke_df['habitat_name'].fillna('None')

type_habitat_counts = poke_df.groupby(['type_1', 'habitat_name']).size().reset_index(name='count')

all_types = poke_df['type_1'].unique()
all_habitats = poke_df['habitat_name'].unique()

all_combinations = pd.MultiIndex.from_product([all_types, all_habitats], names=['type_1', 'habitat_name'])
type_habitat_counts = type_habitat_counts.set_index(['type_1', 'habitat_name']).reindex(all_combinations, fill_value=0).reset_index()
types_and_habitats_fig = px.scatter(type_habitat_counts, 
                 x='type_1', 
                 y='habitat_name', 
                 size='count',  
                 color='type_1', 
                 title="2D Scatter Plot of Pokémon Primary Types vs. Habitats",
                 labels={'type_1': 'Pokémon Type', 'habitat_name': 'Habitat', 'count': 'Pokémon Count'},
                 color_discrete_map=custom_colors, 
                 opacity=0.7)

types_and_habitats_fig.show()
types_and_habitats_fig.write_html("../../visuals/html_files/pokemon_types_vs_habitats.html")



In [14]:
# Group by 'type_1' and count occurrences
type_counts = poke_df.groupby('type_1').size().reset_index(name='count')

# Sort by the 'count' column to make it visually easier to interpret (optional)
type_counts = type_counts.sort_values(by='count', ascending=False)

# ---- Bar Chart ----
fig_bar = px.bar(type_counts, x='type_1', y='count', 
                 labels={'type_1': 'Pokemon Type', 'count': 'Number of Pokémon'},
                 color = 'type_1',
                 color_discrete_map = custom_colors,
                 title="Count of Pokémon by Primary Type")

# Show the bar chart
fig_bar.show()

# ---- Heatmap ----
# Pivot the data for heatmap-style visualization
type_counts_pivot = type_counts.set_index('type_1').T  # Transpose for heatmap format

# Create the heatmap
fig_heatmap = go.Figure(data=go.Heatmap(
                        z=type_counts_pivot.values,
                        x=type_counts_pivot.columns,
                        y=type_counts_pivot.index,
                        colorscale='RdBu_r',  # You can choose other color scales
                        colorbar=dict(title='Number of Pokémon')
))

fig_heatmap.update_layout(
    title="Heatmap of Pokémon Primary Type Counts",
    xaxis_title="Pokemon Type",
    yaxis_title="Count"
)

# Show the heatmap
fig_heatmap.show()


NameError: name 'go' is not defined

In [ ]:
# Group by 'type_1' and count occurrences
type_counts = poke_df.groupby('type_2').size().reset_index(name='count')

# Sort by the 'count' column to make it visually easier to interpret (optional)
type_counts = type_counts.sort_values(by='count', ascending=False)

# ---- Bar Chart ----
fig_bar = px.bar(type_counts, x='type_2', y='count', 
                 labels={'type_2': 'Pokemon Type', 'count': 'Number of Pokémon'},
                 color = 'type_2',
                 color_discrete_map = custom_colors,
                 title="Count of Pokémon by Secondary Type")

# Show the bar chart
fig_bar.show()

# ---- Heatmap ----
# Pivot the data for heatmap-style visualization
type_counts_pivot = type_counts.set_index('type_2').T  # Transpose for heatmap format

# Create the heatmap
fig_heatmap = go.Figure(data=go.Heatmap(
                        z=type_counts_pivot.values,
                        x=type_counts_pivot.columns,
                        y=type_counts_pivot.index,
                        colorscale='RdBu_r',  # You can choose other color scales
                        colorbar=dict(title='Number of Pokémon')
))

fig_heatmap.update_layout(
    title="Heatmap of Pokémon Primary Type Counts",
    xaxis_title="Pokemon Type",
    yaxis_title="Count"
)

# Show the heatmap
fig_heatmap.show()


In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '2'

In [38]:
# 3 Things to note:
# 1. Pokemon ID 678 has a corrupted image, and thus cannot be processed. 
# 2. The process of removing the background removes all white pixels, and as such no Pokemon has white as the dominant color. 
# 3. Despite having set the 'OMP_NUM_THREADS' to 2, the UserWarning recommending it to be done so still appears. 

import requests
import numpy as np
from sklearn.cluster import KMeans
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

# Function to fetch the image from the URL and extract the dominant color
def get_main_color_from_url(image_url):
    try:
        # Fetch the image from the URL
        response = requests.get(image_url)
        img = Image.open(BytesIO(response.content))
        
        # Convert image to RGB (if it has an alpha channel, this will remove transparency)
        img_rgb = img.convert('RGBA')  # Convert to RGBA first
        img_rgb = img_rgb.convert('RGB')  # Then convert to RGB
        
        # Convert image to numpy array
        img_array = np.array(img_rgb)
        
        # Threshold to detect white background (pure white pixels)
        lower_white = np.array([200, 200, 200])  # Lower threshold for white
        upper_white = np.array([255, 255, 255])  # Upper threshold for white
        
        # Create a mask to find white pixels (background)
        mask = np.all(np.logical_and(img_array >= lower_white, img_array <= upper_white), axis=-1)
        
        # Invert mask to focus on the Pokémon (non-background pixels)
        mask_inv = ~mask
        
        # Mask the image to isolate the Pokémon (remove white background)
        img_no_bg = img_array[mask_inv]
        
        # Apply K-means clustering to find the dominant color from the remaining pixels
        kmeans = KMeans(n_clusters=1, random_state=0).fit(img_no_bg)
        
        # The center of the cluster is the main color
        main_color = kmeans.cluster_centers_[0]
        
        # Return the main color as an integer tuple
        return tuple(main_color.astype(int))
    except Exception as e:
        print(f"Error processing {image_url}: {e}")
        return None

# Function to fetch dominant colors concurrently
def get_dominant_colors_concurrently(urls):
    # Create a thread pool executor
    with ThreadPoolExecutor(max_workers=10) as executor:
        # Use executor.map to apply get_main_color_from_url to each URL in the list
        result = list(executor.map(get_main_color_from_url, urls))
    return result

# Assuming poke_df is your dataframe containing URLs in the 'pokemon_portrait' column
# Example of loading a dataframe (this is just for illustration)
# poke_df = pd.read_csv('path_to_pokemon_data.csv')

# Step 1: Get the list of URLs from the 'pokemon_portrait' column
urls = poke_df['pokemon_portrait'].tolist()

# Step 2: Use ThreadPoolExecutor to get the dominant colors concurrently
pokemon_colors = get_dominant_colors_concurrently(urls)

# Step 3: Assign the results to a new column 'pokemon_color'
poke_df['pokemon_color'] = pokemon_colors

# Step 4: Optionally, display the first few rows of the dataframe to see the result
print(poke_df[['name', 'pokemon_color']].head())


Error processing https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/678.png: cannot identify image file <_io.BytesIO object at 0x00000263347B68E0>


c:\Users\chang\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.

c:\Users\chang\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.

c:\Users\chang\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.

c:\Users\chang\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than avai

         name pokemon_color
1   bulbasaur   (7, 12, 10)
2     ivysaur  (14, 22, 19)
3    venusaur  (41, 55, 42)
4  charmander    (17, 9, 4)
5  charmeleon   (26, 11, 8)


In [27]:
display(poke_df.iloc[677,:])

name                                                             meowstic
url                        https://pokeapi.co/api/v2/pokemon-species/678/
generation                                                              6
habitat_name                                                         None
pokemon_description     When in danger, it raises its ears and release...
type_1                                                            psychic
type_2                                                               None
pokemon_portrait        https://raw.githubusercontent.com/PokeAPI/spri...
hp_stat                                                                74
attack_stat                                                            48
defense_stat                                                           76
special_attack_stat                                                    83
special_defense_stat                                                   81
speed_stat                            

In [34]:
import pandas as pd
import numpy as np
import matplotlib.colors as mcolors

# Function to classify colors into perceptual families
def classify_color(rgb):
    if rgb is None:  # Handle None values gracefully
        return 'Unknown'
    
    # Ensure the RGB values are valid (i.e., they are not None and are within range)
    if len(rgb) != 3 or any(val < 0 or val > 255 for val in rgb):
        return 'Unknown'
    
    # Convert RGB to HSV (Hue, Saturation, Value)
    rgb_normalized = np.array(rgb) / 255.0
    hsv = mcolors.rgb_to_hsv(rgb_normalized)
    
    hue = hsv[0] * 360  # Hue in degrees (0-360)
    saturation = hsv[1]
    value = hsv[2]
    
    # Classify based on hue ranges and saturation/value thresholds
    if saturation < 0.2:  # Low saturation means it's likely grayish or brownish
        return 'Gray' if value > 0.5 else 'Brown'
    elif 0 <= hue < 30 or 330 <= hue < 360:
        return 'Red'
    elif 30 <= hue < 60:
        return 'Orange'
    elif 60 <= hue < 120:
        return 'Yellow'
    elif 120 <= hue < 180:
        return 'Green'
    elif 180 <= hue < 240:
        return 'Cyan'
    elif 240 <= hue < 300:
        return 'Blue'
    elif 300 <= hue < 330:
        return 'Purple'
    elif 330 <= hue < 360:
        return 'Pink'
    else:
        return 'Unknown'  # For any undefined cases

# Apply the classify_color function to each row in the 'pokemon_color' column
poke_df['color_family'] = poke_df['pokemon_color'].apply(classify_color)

# Now poke_df will have a new column 'color_family' with perceptual color categories
print(poke_df[['pokemon_color', 'color_family']].head())  # Print the first few rows


  pokemon_color color_family
1   (7, 12, 10)        Green
2  (14, 22, 19)        Green
3  (41, 55, 42)        Green
4    (17, 9, 4)          Red
5   (26, 11, 8)          Red


In [36]:
poke_df['color_family'].value_counts()

color_family
Red        250
Orange     188
Brown      162
Cyan       146
Yellow     102
Blue        79
Purple      40
Green       39
Gray        18
Unknown      1
Name: count, dtype: int64